# <b>Right Left Recognition Algorithm</b>

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

def process_frame(frame):
    height, width, _ = frame.shape
    roi_height = int(height / 3)
    roi_top = height - roi_height
    roi = frame[roi_top:, :]

    cv2.line(roi, (width // 2, 0), (width // 2, roi_height), (0, 255, 0), 2)

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower_white = np.array([0, 0, 200], dtype=np.uint8)
    upper_white = np.array([255, 30, 255], dtype=np.uint8)
    white_mask = cv2.inRange(hsv, lower_white, upper_white)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 윤곽선 검출 및 처리 
    if len(contours) >= 1:
        max_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(roi, [max_contour], -1, (0, 255, 0), 2)

        # 물체의 중심 계산 및 결과 출력 
        M = cv2.moments(max_contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            
            center_line = width // 2
            if cx < center_line - 50:
                return "LEFT"
            elif cx > center_line + 50:
                return "RIGHT"

    return None

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def update_video_display(cap, image_widget, text_widget):
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Video file ended or error occurred")
            break

        result = process_frame(frame)
        
        if result:
            text_widget.value = result
        
        result_frame = frame.copy()
        height, width, _ = result_frame.shape
        roi_height = int(height / 3)
        roi_top = height - roi_height
        roi = result_frame[roi_top:, :]
        result_frame[roi_top:, :] = roi

        original_bytes = convert_to_bytes(frame)
        result_bytes = convert_to_bytes(result_frame)
        
        image_widget.value = original_bytes
        
        clear_output(wait=True)
        display(widgets.HBox([image_widget, text_widget]))

        time.sleep(0.03)

cap = cv2.VideoCapture('curve_video.avi')

image_widget = widgets.Image(format='jpeg')
text_widget = widgets.Label(value='')

update_video_display(cap, image_widget, text_widget)

cap.release()


# <b>Right Left Recognition Algorithm With Motor Movement</b>

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
from tiki.mini import TikiMini

tiki = TikiMini()
tiki.set_motor_mode(tiki.MOTOR_MODE_PID)

def process_frame(frame):
    height, width, _ = frame.shape
    roi_height = int(height / 3)
    roi_top = height - roi_height
    roi = frame[roi_top:, :]

    cv2.line(roi, (width // 2, 0), (width // 2, roi_height), (0, 255, 0), 2)

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower_white = np.array([0, 0, 200], dtype=np.uint8)
    upper_white = np.array([255, 30, 255], dtype=np.uint8)
    white_mask = cv2.inRange(hsv, lower_white, upper_white)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 윤곽선 검출 및 처리 
    if len(contours) >= 1:
        max_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(roi, [max_contour], -1, (0, 255, 0), 2)

        # 물체의 중심 계산 및 결과 출력 
        M = cv2.moments(max_contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            
            center_line = width // 2
            if cx < center_line - 50:
                return "LEFT"
            elif cx > center_line + 50:
                return "RIGHT"

    return None

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def update_video_display(cap, image_widget, text_widget):
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Video file ended or error occurred")
            break

        result = process_frame(frame)
        
        if result:
            text_widget.value = result
            if result == "LEFT":
                tiki.counter_clockwise(30)
                tiki.log("LEFT")
                time.sleep(2)
            elif result == "RIGHT":
                tiki.clockwise(30)
                tiki.log("RIGHT")
                time.sleep(2)
        
        result_frame = frame.copy()
        height, width, _ = result_frame.shape
        roi_height = int(height / 3)
        roi_top = height - roi_height
        roi = result_frame[roi_top:, :]
        result_frame[roi_top:, :] = roi

        original_bytes = convert_to_bytes(frame)
        result_bytes = convert_to_bytes(result_frame)
        
        image_widget.value = original_bytes
        
        clear_output(wait=True)
        display(widgets.HBox([image_widget, text_widget]))

        time.sleep(0.03)

cap = cv2.VideoCapture('curve_video.avi')

image_widget = widgets.Image(format='jpeg')
text_widget = widgets.Label(value='')

update_video_display(cap, image_widget, text_widget)

cap.release()
